In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_enAsCas12a_HF1():
    """
    Loads (60,4) data from 'Feature_guide_baseonly_PAM_complete_enAsCas12a_HF1_TTTV_filtered_reduced_seq_feature.txt'.
    """
    data_branch1 = []
    current_array = []
    with open('Feature_guide_baseonly_PAM_complete_enAsCas12a_HF1_TTTV_filtered_reduced_seq_feature.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1


def load_branch2_data_enAsCas12a_HF1():
    """
    Loads (90,3) data from 'Feature_guide_target_complete_struct_only_bhah_enAsCas12a_HF1_TTTV_filtered_KD_reduced_feature_model.txt'.
    """
    data_branch2 = []
    current_array = []
    with open('Feature_guide_target_complete_struct_only_bhah_enAsCas12a_HF1_TTTV_filtered_KD_reduced_feature_model.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch2.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch2.append(current_array)
    X_branch2 = np.array(data_branch2)
    X_branch2 = X_branch2.reshape(len(X_branch2), 90, 3)
    return X_branch2


def load_branch3_data_enAsCas12a_HF1(filename="Feature_guide_target_complete_value_features_120D_all_positive_enAsCas12a_HF1_TTTV_filtered_KD_reduced_feature_model.txt"):
    """
    Loads 120-dimensional numeric features from file.
    Each line has 120 floats (space- or comma-delimited).
    """
    X_branch3 = []
    with open(filename, 'r') as file:
        for line in file:
            line = line.strip()
            # Expect 120 numbers per line
            values = line.replace('[', '').replace(']', '').split()
            float_vals = [float(v.strip().replace(',', '')) for v in values]
            X_branch3.append(float_vals)
    
    X_branch3 = np.array(X_branch3)  # shape: (n_samples, 120)
    X_branch3 = X_branch3.reshape(len(X_branch3), 120)
    print("X_branch3 shape:", X_branch3.shape)
    return X_branch3


def load_reaction_rates_enAsCas12a_HF1():
    with open('enAsCas12a_HF1_indel_frequency_TTTV_filtered.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


########################################
# 2. Graph Data Utilities (for GNN)
########################################

def matrix_to_edge_index(prob_matrix):
    edges_source = []
    edges_target = []
    L = len(prob_matrix)
    for i in range(L):
        for j in range(i + 1, L):  # only i < j
            if prob_matrix[i][j] > 0.01:
                edges_source.append(i)
                edges_target.append(j)
    return [edges_source, edges_target]

def matrix_to_edge_index_collapsed(prob_matrix: np.ndarray, threshold: float = 0.01):
    """
    Build undirected edges (i < j) from a dense probability matrix using a threshold.
    Returns edge_index as [sources, targets] lists of ints.
    """
    src, dst = [], []
    L = prob_matrix.shape[0]
    for i in range(L):
        for j in range(i+1, L):
            if prob_matrix[i, j] > threshold:
                src.append(i)
                dst.append(j)
    return [src, dst]

def generate_edge_features(edge_index, prob_matrix):
    if not edge_index:
        return []
    feats = []
    for i in range(len(edge_index[0])):
        src = edge_index[0][i]; tgt = edge_index[1][i]
        feats.append([prob_matrix[src][tgt]])  # shape [E, 1]
    return feats

def _num_nodes_from_edge_index(edge_index_list, fallback):
    if not edge_index_list or len(edge_index_list[0]) == 0:
        return fallback
    mx = 0
    for a in edge_index_list:
        if len(a) > 0:
            mx = max(mx, max(a))
    return mx + 1


def collapse_to_last_k_plus_1_sum_enAsCas12a_HF1(prob_matrix: np.ndarray, last_k: int = 20) -> np.ndarray:
    """
    last_k refers to the spacer length
    Collapse an LxL base-pair probability matrix into (last_k + 1) x (last_k + 1):
      - Rows/cols last_k..38: original last_k_base..39
      - Row/col 0: last_k: an 'outside' super-node representing bases <= last_k
    For each i in [last_k..38], the edge prob to the super-node is the
    SUM of probabilities of pairing to bases <= last_k.
    """
    L = prob_matrix.shape[0]
    K = last_k
    new_size = K + 1
    newP = np.zeros((new_size, new_size), dtype=float)

    newP[:K, :K] = prob_matrix[(L-K):, (L-K):]

    if L > K:
        # Outside block (0..K-1)
        outside_block = prob_matrix[(L-K):L, :(L-K)]  # shape: K x (L-K)
        # Sum of probabilities across outside bases
        p_sum = np.sum(outside_block, axis=1)
        # Fill symmetric connections to the super-node K
        newP[:K, K] = p_sum
        newP[K, :K] = p_sum

    # Leave newP[K, K] as 0
    return newP

def slice_and_renumber_duplex_enAsCas12a_HF1(prob_matrix, len_guide, len_target, k=20):
    """
    k refers to the spacer length
    Keep only the last k positions of the guide and the last k positions of the target,
    Returns a (kg+kt) x (kg+kt) reduced/renumbered matrix.
    """
    L = len_guide + len_target
    assert prob_matrix.shape == (L, L), "prob_matrix size mismatch."

    kg = min(k, len_guide)
    kt = min(k, len_target)

    # Original indices: guide = [0..len_guide-1], target = [len_guide..L-1]
    sel_g = np.arange(len_guide - kg, len_guide)  # last k of guide
    sel_t = np.arange(L - kt, L)                  # last k of target
    S = np.concatenate([sel_g, sel_t])

    # Extract submatrix for these rows/cols
    P = prob_matrix[np.ix_(S, S)].copy()

    # Reverse target rows/cols
    P[kg:, :] = P[kg:, :][::-1, :]
    P[:, kg:] = P[:, kg:][:, ::-1]

    return P

# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

def load_dna_reaction_data_enAsCas12a_HF1():
    with open('enAsCas12a_HF1_full_guide_sequences_TTTV_filtered.txt', 'r') as f:
        crna_sequences = [line.strip() for line in f.readlines()]
    with open('enAsCas12a_HF1_target_sequences_noPAM_TTTV_filtered.txt', 'r') as f:
        target_sequences = [line.strip() for line in f.readlines()]

    LAST_K = 20  # keep last 20 bases, collapse the rest into node index 20 (21st base)
        
    crna_edge_indices = []
    crna_edge_features = []
    for guide in crna_sequences:
        prob_full = pairs(strands=guide, model=my_model_RNA).to_array()
        prob_collapsed = collapse_to_last_k_plus_1_sum_enAsCas12a_HF1(prob_full, last_k=LAST_K)
        edge_index = matrix_to_edge_index_collapsed(prob_collapsed, threshold=0.01)
        feats = generate_edge_features(edge_index, prob_collapsed)
        crna_edge_indices.append(edge_index)   # each is [sources, targets] with 0..20 node indices
        crna_edge_features.append(feats)       # [[p_ij], ...]

    ssDNA_target_bh_edge_indices = []
    ssDNA_target_bh_edge_features = []
    for i in range (0, len(crna_sequences)):
        prob_matrix = pairs(strands=target_sequences[i], model=my_model_DNA).to_array()
        prob_matrix = prob_matrix[::-1, ::-1]
        edge_index = matrix_to_edge_index(prob_matrix)
        feats = generate_edge_features(edge_index, prob_matrix)
        ssDNA_target_bh_edge_indices.append(edge_index)
        ssDNA_target_bh_edge_features.append(feats)
    
    duplex_edge_indices = []
    duplex_edge_features = []
  
    for i in range (0, len(crna_sequences)):
        prob_matrix = pairs(strands=[crna_sequences[i], target_sequences[i]], model=my_model_RNA).to_array()
        len_g, len_t = len(crna_sequences[i]), len(target_sequences[i])
        # Crop to guide[19..39] + target[last 20], renumber both
        updated_P = slice_and_renumber_duplex_enAsCas12a_HF1(prob_matrix, len_g, len_t, k=20)
        edge_index = matrix_to_edge_index(updated_P)
        feats = generate_edge_features(edge_index, updated_P)
        duplex_edge_indices.append(edge_index)
        duplex_edge_features.append(feats)

    with open('enAsCas12a_HF1_indel_frequency_TTTV_filtered.txt', 'r') as f:
        reaction_rates = [float(line.strip()) for line in f.readlines()]
    
    return (crna_edge_indices, ssDNA_target_bh_edge_indices, duplex_edge_indices, 
            crna_edge_features, ssDNA_target_bh_edge_features, duplex_edge_features, 
            reaction_rates)
    



from typing import Iterable, List, Tuple, Any

# Each graph set is a tuple:
# (crna_ei, tbh_ei, d_ei, crna_ea, tbh_ea, d_ea, y)
GraphSet = Tuple[Iterable[Any], Iterable[Any], Iterable[Any],
                 Iterable[Any], Iterable[Any], Iterable[Any], Iterable[Any]]

def combine_dna_graph_sets(*graph_sets: GraphSet):
    """Concatenate any number of DNA/RNA graph sets along the sample axis.
       Optionally return a source label per sample (0,1,2,...) indicating origin."""
    if not graph_sets:
        raise ValueError("Provide at least one graph set")

    c_ei: List[Any] = []
    tbh_ei: List[Any] = []
    d_ei: List[Any] = []
    c_ea: List[Any] = []
    tbh_ea: List[Any] = []
    d_ea: List[Any] = []
    y: List[Any] = []

    for sid, gs in enumerate(graph_sets):
        try:
            c_ei_a, tbh_ei_a, d_ei_a, c_ea_a, tbh_ea_a, d_ea_a, y_a = gs
        except Exception as e:
            raise ValueError(f"Graph set #{sid} must be a 7-tuple") from e

        # Optional consistency check per set
        n = len(y_a)
        if not all(len(lst) == n for lst in (c_ei_a, tbh_ei_a, d_ei_a, c_ea_a, tbh_ea_a, d_ea_a)):
            raise ValueError(f"Graph set #{sid} has mismatched lengths")

        c_ei.extend(list(c_ei_a))
        tbh_ei.extend(list(tbh_ei_a))
        d_ei.extend(list(d_ei_a))
        c_ea.extend(list(c_ea_a))
        tbh_ea.extend(list(tbh_ea_a))
        d_ea.extend(list(d_ea_a))
        y.extend(list(y_a))

    combined = (c_ei, tbh_ei, d_ei, c_ea, tbh_ea, d_ea, y)
    return combined


class CRNADuplexSubstructuresDataset(Dataset):
    def __init__(self, crna_edge_indices, ssDNA_target_bh_edge_indices, duplex_edge_indices, 
                 crna_edge_features, ssDNA_target_bh_edge_features, duplex_edge_features, 
                 reaction_rates):
        super().__init__()
        self.crna_edge_indices = crna_edge_indices
        self.ssDNA_target_bh_edge_indices = ssDNA_target_bh_edge_indices
        self.duplex_edge_indices = duplex_edge_indices
        self.crna_edge_features = crna_edge_features
        self.ssDNA_target_bh_edge_features = ssDNA_target_bh_edge_features
        self.duplex_edge_features = duplex_edge_features
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)
    def __len__(self):
        return self.num_samples
    def __getitem__(self, idx):
        c_edge_index_list = self.crna_edge_indices[idx]
        if not c_edge_index_list:
            c_edge_index = torch.empty((2,0), dtype=torch.long)
            n_c = 20  # fallback (e.g., 20-nt)
        else:
            c_edge_index = torch.tensor(c_edge_index_list, dtype=torch.long)
            n_c = _num_nodes_from_edge_index(c_edge_index_list, fallback=20)
        c_x = torch.zeros((n_c, 4), dtype=torch.float)  # placeholder
        c_edge_attr = torch.tensor(self.crna_edge_features[idx], dtype=torch.float)
        if c_edge_attr.dim() == 1:
            c_edge_attr = c_edge_attr.unsqueeze(1)   # [E,1]
        cRNA_data = Data(x=c_x, edge_index=c_edge_index, edge_attr=c_edge_attr)

        t_bh_edge_index_list = self.ssDNA_target_bh_edge_indices[idx]
        if not t_bh_edge_index_list:
            t_bh_edge_index = torch.empty((2,0), dtype=torch.long)
            n_t = 20
        else:
            t_bh_edge_index = torch.tensor(t_bh_edge_index_list, dtype=torch.long)
            n_t = _num_nodes_from_edge_index(t_bh_edge_index_list, fallback=20)
        t_bh_x = torch.zeros((n_t, 4), dtype=torch.float)
        t_bh_edge_attr = torch.tensor(self.ssDNA_target_bh_edge_features[idx], dtype=torch.float)
        if t_bh_edge_attr.dim() == 1:
            t_bh_edge_attr = t_bh_edge_attr.unsqueeze(1)   # [E,1]
        ssDNA_target_bh_data = Data(x=t_bh_x, edge_index=t_bh_edge_index, edge_attr=t_bh_edge_attr)
        
        d_edge_index_list = self.duplex_edge_indices[idx]
        if not d_edge_index_list:
            d_edge_index = torch.empty((2,0), dtype=torch.long)
            n_d = 40
        else:
            d_edge_index = torch.tensor(d_edge_index_list, dtype=torch.long)
            n_d = _num_nodes_from_edge_index(d_edge_index_list, fallback=40)
        d_x = torch.zeros((n_d, 4), dtype=torch.float)
        d_edge_attr = torch.tensor(self.duplex_edge_features[idx], dtype=torch.float)
        if d_edge_attr.dim() == 1:
            d_edge_attr = d_edge_attr.unsqueeze(1)   # [E,1]
        duplex_data = Data(x=d_x, edge_index=d_edge_index, edge_attr=d_edge_attr)
        
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return cRNA_data, ssDNA_target_bh_data, duplex_data, y_val


# 1) Replace the old .npy loader with your TXT file loader
def load_teacher_preds(path="HTCas9_HT11_RfxCas13d_change_seq_sampled_2_tiger_iMeta_combined_preds_enAsCas12a_HF1.txt"):
    """
    Load teacher predictions for NEW inputs (one float per line).
    Returns: np.ndarray of shape [N], dtype float32
    """
    return np.loadtxt(path, dtype=np.float32)


########################################
# 3. CNN, GNN, MLP Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x
        

class CNNBranch2(nn.Module):
    """
    CNN branch for (90,3).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(3, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*45, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,3,45)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,45)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import NNConv, global_mean_pool, global_max_pool


class GNNBranch(nn.Module):
    def __init__(self, hidden_dim=32, edge_attr_dim=1):
        super().__init__()
        self.hidden_dim = hidden_dim

        def make_edge_mlp(edge_attr_dim, in_channels, out_channels):
            return nn.Sequential(
                nn.Linear(edge_attr_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, in_channels * out_channels)
            )

        self.edge_mlp_c   = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_c2  = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_c = NNConv(4, hidden_dim, self.edge_mlp_c, aggr='mean')
        self.conv2_c = NNConv(hidden_dim, hidden_dim, self.edge_mlp_c2, aggr='mean')

        self.edge_mlp_bh  = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_bh2 = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_t_bh = NNConv(4, hidden_dim, self.edge_mlp_bh, aggr='mean')
        self.conv2_t_bh = NNConv(hidden_dim, hidden_dim, self.edge_mlp_bh2, aggr='mean')

        self.edge_mlp_d   = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_d2  = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_d = NNConv(4, hidden_dim, self.edge_mlp_d, aggr='mean')
        self.conv2_d = NNConv(hidden_dim, hidden_dim, self.edge_mlp_d2, aggr='mean')

        self.mlp_merge = nn.Sequential(
            nn.Linear(hidden_dim * 3 * 2, hidden_dim),  # 3 graphs, mean+max
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def pool(self, x, batch):
        return torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)

    def forward(self, crna_data, ssDNA_target_bh_data, duplex_data):
        def process(graph_data, conv1, conv2):
            x, edge_index, edge_attr, batch = graph_data.x, graph_data.edge_index, graph_data.edge_attr, graph_data.batch
            x = F.elu(conv1(x, edge_index, edge_attr))   # << correct order
            x = F.elu(conv2(x, edge_index, edge_attr))
            return self.pool(x, batch)

        x_c   = process(crna_data, self.conv1_c,   self.conv2_c)
        x_tbh = process(ssDNA_target_bh_data, self.conv1_t_bh, self.conv2_t_bh)
        x_d   = process(duplex_data, self.conv1_d, self.conv2_d)

        merged = torch.cat([x_c, x_tbh, x_d], dim=1)
        return self.mlp_merge(merged)


class MLPBranch120(nn.Module):
    """
    A single hidden layer MLP for 120D => output dimension mlp_dim.
    No separate "output" layer; just one fc + ReLU => final embedding.
    """
    def __init__(self, input_dim=120, hidden_dim=32):
        super().__init__()
        self.fc = nn.Linear(input_dim, hidden_dim)
    def forward(self, x):
        # x shape: (batch,120)
        x = F.relu(self.fc(x))  # (batch,hidden_dim)
        return x

########################################
# 4. Final Fusion Model
########################################

class CNN_GNN_MLP_Fusion(nn.Module):
    """
    End-to-end: 
      - CNNBranch1 => feat1
      - CNNBranch2 => feat2
      - GNNBranch  => feat_gnn
      - MLPBranch120 => feat_mlp
    Concat => dropout => final FC => 1
    """
    def __init__(self,
                 filters1, kernel_size1, dense_units1,  # CNN1
                 filters2, kernel_size2, dense_units2,  # CNN2
                 gnn_hidden_dim,
                 mlp_hidden_dim,  # single hidden dimension for MLP
                 final_fc_dim,
                 dropout_rate=0.0):  # new hyperparameter for dropout
        super().__init__()
        
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.cnn_branch2 = CNNBranch2(filters2, kernel_size2, dense_units2)
        self.gnn_branch = GNNBranch(hidden_dim=gnn_hidden_dim)
        self.mlp_branch = MLPBranch120(input_dim=120, hidden_dim=mlp_hidden_dim)
        
        # total dimension = (dense_units1 + dense_units2 + gnn_hidden_dim + mlp_hidden_dim)
        total_dim = dense_units1 + dense_units2 + gnn_hidden_dim + mlp_hidden_dim
        
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(total_dim, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    
    def forward(self, crna_data, ssDNA_target_bh_data, duplex_data, x1, x2, x3):
        feat1 = self.cnn_branch1(x1)                   # (batch, dense_units1)
        feat2 = self.cnn_branch2(x2)                   # (batch, dense_units2)
        feat_gnn = self.gnn_branch(crna_data, ssDNA_target_bh_data, duplex_data)  # (batch, gnn_hidden_dim)
        feat_mlp = self.mlp_branch(x3)                 # (batch, mlp_hidden_dim)
        
        merged = torch.cat([feat1, feat2, feat_gnn, feat_mlp], dim=1)
        # Apply dropout on the concatenated features
        merged = F.dropout(merged, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))   # (batch, final_fc_dim)
        out = self.out(x)                 # (batch, 1)
        return out.view(-1)



########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, graph_dataset, X1, X2, X3, reaction_rates):
        super().__init__()
        self.graph_dataset = graph_dataset
        self.X1 = X1
        self.X2 = X2
        self.X3 = X3
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)
        
        assert len(graph_dataset) == self.num_samples
        assert len(X1) == self.num_samples
        assert len(X2) == self.num_samples
        assert len(X3) == self.num_samples
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        crna_data, ssDNA_target_bh_data, duplex_data, y_val = self.graph_dataset[idx]
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)  
        x2 = torch.tensor(self.X2[idx], dtype=torch.float)  
        x3 = torch.tensor(self.X3[idx], dtype=torch.float)  
        return crna_data, ssDNA_target_bh_data, duplex_data, x1, x2, x3, y_val

def hybrid_collate(batch):
    from torch_geometric.data import Batch
    cRNA_list, ssDNA_target_bh_list, duplex_list, x1_list, x2_list, x3_list, y_list = zip(*batch)
    batch_cRNA = Batch.from_data_list(cRNA_list)
    batch_ssDNA_target_bh = Batch.from_data_list(ssDNA_target_bh_list)
    batch_duplex = Batch.from_data_list(duplex_list)
    x1 = torch.stack(x1_list, dim=0)
    x2 = torch.stack(x2_list, dim=0)
    x3 = torch.stack(x3_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return batch_cRNA, batch_ssDNA_target_bh, batch_duplex, x1, x2, x3, y

# ==== Loss factory (choose MSE or Huber) ====
def make_loss(loss_name: str):
    if loss_name.lower() == "mse":
        return nn.MSELoss()
    elif loss_name.lower() == "huber":
        return nn.SmoothL1Loss(beta=1.0)
    else:
        raise ValueError(f"Unknown loss '{loss_name}'")


# ==== NEW: fine-tune student on real labels (MSE to experimental y), early stopping patience=10 ====
def fine_tune_on_real(model,
                      train_loader,
                      val_loader=None,
                      epochs=1000,
                      lr=5e-4,
                      weight_decay=1e-5,
                      patience=10,
                      device="cuda",
                      freeze_backbone=False,
                      loss_fn=None,
                      # NEW:
                      ft_loss_weight: float = 2.0,              # Option 2
                      aux_loader = None,                        # Optional: a second loader
                      alpha_schedule: str = "constant",         # "constant" | "linear_warmup"
                      alpha_warmup_epochs: int = 0):            # how long to ramp FT up
    """
    If aux_loader is provided, total_loss = (1-alpha)*aux_loss + alpha*(ft_loss_weight * ft_loss).
    alpha schedule:
      - constant: alpha = 1.0
      - linear_warmup: alpha = min(1, epoch / alpha_warmup_epochs) if >0 else 1.0
    """
    model = model.to(device)

    if freeze_backbone:
        for m in [model.cnn_branch1, model.cnn_branch2, model.gnn_branch, model.mlp_branch]:
            for p in m.parameters(): p.requires_grad = False
        params = list(model.hidden.parameters()) + list(model.out.parameters())
        opt = optim.Adam(params, lr=lr, weight_decay=weight_decay)
    else:
        opt = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    loss_fn = loss_fn or nn.MSELoss()
    best_model_state, best_val = None, float("inf")
    counter = 0

    # if aux is provided, make an iterator we can cycle
    if aux_loader is not None:
        from itertools import cycle
        aux_iter = cycle(aux_loader)

    for epoch in range(1, epochs+1):
        model.train()
        # compute alpha
        if alpha_schedule == "linear_warmup" and alpha_warmup_epochs > 0:
            alpha = min(1.0, epoch / float(alpha_warmup_epochs))
        else:
            alpha = 1.0

        total_loss_epoch = 0.0
        for batch in train_loader:
            c,tbh,d,x1,x2,x3,y = batch
            c,tbh,d = c.to(device), tbh.to(device), d.to(device)
            x1,x2,x3,y = x1.to(device), x2.to(device), x3.to(device), y.to(device)

            pred = model(c,tbh,d,x1,x2,x3)
            ft_loss = loss_fn(pred, y)

            # Optional auxiliary loss
            if aux_loader is not None:
                c2,tbh2,d2,x12,x22,x32,y2 = next(aux_iter)
                c2,tbh2,d2 = c2.to(device), tbh2.to(device), d2.to(device)
                x12,x22,x32,y2 = x12.to(device), x22.to(device), x32.to(device), y2.to(device)
                pred2 = model(c2,tbh2,d2,x12,x22,x32)
                aux_loss = loss_fn(pred2, y2)   # replace with KD loss if you want
                loss = (1.0 - alpha) * aux_loss + alpha * (ft_loss_weight * ft_loss)
            else:
                loss = ft_loss_weight * ft_loss

            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss_epoch += float(loss.item())

        total_loss_epoch /= max(1, len(train_loader))
        msg = f"[FT] Epoch {epoch} | Train Loss (weighted): {total_loss_epoch:.6f}"

        if val_loader is not None:
            model.eval()
            total_val = 0.0
            with torch.no_grad():
                for c,tbh,d,x1,x2,x3,y in val_loader:
                    c,tbh,d = c.to(device), tbh.to(device), d.to(device)
                    x1,x2,x3,y = x1.to(device), x2.to(device), x3.to(device), y.to(device)
                    pred = model(c,tbh,d,x1,x2,x3)
                    total_val += loss_fn(pred, y).item()
            total_val /= max(1, len(val_loader))
            msg += f" | Val MSE: {total_val:.6f}"
            if total_val < best_val:
                best_val = total_val
                best_model_state = copy.deepcopy(model)
                counter = 0
            else:
                counter += 1

            if counter >= patience:
                print(msg + " | Early stopping (patience reached)")
                break

        print(msg)

    if best_model_state is not None:
        model = best_model_state
    return model, best_val



# ===== Global, load-once FT cache =====
FT_CACHE = {}

def prepare_ft_cache(random_state=42):
    """Load FT data ONCE and store in FT_CACHE."""
    global FT_CACHE
    if FT_CACHE.get("_ready"):
        return  # already prepared

    # ----- Fine-tune data (real labels) -----
    X1_true = load_branch1_data_enAsCas12a_HF1()
    X2_true = load_branch2_data_enAsCas12a_HF1()
    X3_true = load_branch3_data_enAsCas12a_HF1()
    (c_ei_ht, tbh_ei_ht, d_ei_ht, c_ea_ht, tbh_ea_ht, d_ea_ht, y_true) = load_dna_reaction_data_enAsCas12a_HF1()
    
    y_true = np.asarray(y_true, dtype=np.float32)

    graph_true = CRNADuplexSubstructuresDataset(
        c_ei_ht, tbh_ei_ht, d_ei_ht, c_ea_ht, tbh_ea_ht, d_ea_ht, reaction_rates=y_true
    )

    np.random.seed(42)
    idx = np.arange(len(y_true))
    selected_indices = np.random.choice(len(idx), size=2913, replace=False)
    selected_indices_test = np.random.choice(selected_indices, size=10, replace=False)
    tr_idx, va_idx = train_test_split(selected_indices_test, test_size=0.2, random_state=42)

    tr_idx = np.asarray(tr_idx, dtype=idx.dtype)
    va_idx = np.asarray(va_idx, dtype=idx.dtype)

    FT_CACHE.update({
        "X1_true": X1_true,
        "X2_true": X2_true,
        "X3_true": X3_true,
        "y_true": y_true,
        "graph_true": graph_true,
        "tr_idx": tr_idx,
        "va_idx": va_idx,
        "_ready": True,
    })

from torch.utils.data import WeightedRandomSampler

def build_ft_loaders_from_cache(batch_size, pin=False,
                                repeat_k:int=100,
                                sampler_mode:str="weighted",   # "shuffle" | "weighted"
                                sampler_power:float=1.2):     # how strongly to bias weights (>=1)
    """
    repeat_k: duplicate training set K times (oversampling)
    sampler_mode: "shuffle" (default) or "weighted"
    sampler_power: if >1, accentuates the weighting curve
    """
    X1 = FT_CACHE["X1_true"]; X2 = FT_CACHE["X2_true"]; X3 = FT_CACHE["X3_true"]; yT = FT_CACHE["y_true"]
    gds = FT_CACHE["graph_true"]
    tr_idx = FT_CACHE["tr_idx"]; va_idx = FT_CACHE["va_idx"]

    # ---- repetition on TRAIN only ----
    if repeat_k > 1:
        tr_idx_rep = np.tile(tr_idx, repeat_k)
    else:
        tr_idx_rep = tr_idx

    ft_train = HybridDataset(Subset(gds, tr_idx_rep), X1[tr_idx_rep], X2[tr_idx_rep], X3[tr_idx_rep], yT[tr_idx_rep])
    ft_val   = HybridDataset(Subset(gds, va_idx),     X1[va_idx],     X2[va_idx],     X3[va_idx],     yT[va_idx])


    # ---- sampler (weighted vs shuffle) ----
    if sampler_mode == "weighted":
        # Example rule: emphasize top/bottom labels by distance from median (robust spread).
        y_train = yT[tr_idx_rep]
        med = np.median(y_train)
        spread = np.median(np.abs(y_train - med)) + 1e-8
        base_w = 1.0 + np.abs(y_train - med) / spread   # >= 1
        if sampler_power != 1.0:
            base_w = np.power(base_w, sampler_power)
        weights = base_w.astype(np.float64)
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


        train_loader = DataLoader(ft_train, batch_size=batch_size, sampler=sampler,
                                  collate_fn=hybrid_collate, num_workers=0, pin_memory=pin)
    else:
        train_loader = DataLoader(ft_train, batch_size=batch_size, shuffle=True,
                                  collate_fn=hybrid_collate, num_workers=0, pin_memory=pin)

    val_loader   = DataLoader(ft_val, batch_size=batch_size, shuffle=False,
                              collate_fn=hybrid_collate, num_workers=0, pin_memory=pin)
    return train_loader, val_loader


########################################
# 7. Main Pipeline
########################################
# 5) WHERE TO CALL IT
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    pin = (device == "cuda")

    best_teacher = torch.load(f"trial_{2}_model_single_hidden_mlp_complete_HTCas9_HT11_RfxCas13d_change_seq_sampled_2_tiger_iMeta_combined_L2_DROPOUT_reduced_feature_120D_all_positive_MLP_all.pt", weights_only=False)
    best_teacher.eval()
    

    def ft_objective(trial):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        pin = (device == "cuda")
    
        # ----- Suggest model hyperparams -----
        filters1       = trial.suggest_int("filters1", 32, 128, step=32)
        kernel_size1   = trial.suggest_categorical("kernel_size1", [3,5,7])
        dense_units1   = trial.suggest_int("dense_units1", 64, 256, step=64)
        filters2       = trial.suggest_int("filters2", 32, 128, step=32)
        kernel_size2   = trial.suggest_categorical("kernel_size2", [3,5,7])
        dense_units2   = trial.suggest_int("dense_units2", 64, 256, step=64)
        gnn_hidden_dim = trial.suggest_int("gnn_hidden_dim", 32, 128, step=32)
        mlp_hidden_dim = trial.suggest_int("mlp_hidden_dim", 32, 128, step=32)
        final_fc_dim   = trial.suggest_int("final_fc_dim", 64, 256, step=64)
        dropout_rate   = trial.suggest_float("dropout_rate", 0.0, 0.5)
    
        # ----- Suggest FT hyperparams -----
        ft_lr           = trial.suggest_float("ft_lr", 1e-6, 1e-4, log=True)
        ft_weight_decay = trial.suggest_loguniform("ft_weight_decay", 1e-6, 1e-3)
        ft_batch_size   = trial.suggest_categorical("ft_batch_size", [128, 256, 512])
        freeze_backbone = trial.suggest_categorical("freeze_backbone", [False])
        ft_loss_name    = "mse"
        ft_epochs       = 1000
        ft_patience     = 10

        # === NEW knobs (covering the 5 options) ===
        # (1) Sampler oversampling
        sampler_mode    = trial.suggest_categorical("ft_sampler_mode", ["weighted"])
        sampler_power   = trial.suggest_float("ft_sampler_power", 1.0, 1.5)   # >1 = stronger bias

        # (4) Simple repetition oversampling
        repeat_k        = trial.suggest_categorical("ft_repeat_k", [100])

        # (2) Loss scaling
        ft_loss_weight  = trial.suggest_float("ft_loss_weight", 1.0, 8.0, log=True)

        # (5) Curriculum schedule (only matters if you provide an aux_loader)
        alpha_schedule  = trial.suggest_categorical("alpha_schedule", ["constant", "linear_warmup"])
        alpha_warmup    = trial.suggest_int("alpha_warmup_epochs", 0, 20)

        ft_student = copy.deepcopy(best_teacher).to(device)
        ft_student.eval()

        # ----- Fine-tune data (real labels) -----
        ft_train_loader, ft_val_loader = build_ft_loaders_from_cache(
            ft_batch_size, pin=pin,
            repeat_k=repeat_k,
            sampler_mode=sampler_mode,
            sampler_power=sampler_power
        )

        # If you want to mix in an auxiliary loader (e.g., KD/pretrain), build it here and pass as aux_loader.
        aux_loader = None  # keep None unless you purposely add a second dataset

        # ----- Fine-tune on real -----
        ft_student, ft_val_mse = fine_tune_on_real(
            model=ft_student, train_loader=ft_train_loader, val_loader=ft_val_loader,
            epochs=ft_epochs, lr=ft_lr, weight_decay=ft_weight_decay,
            patience=ft_patience, device=device,
            freeze_backbone=freeze_backbone, loss_fn=make_loss(ft_loss_name),
            # NEW:
            ft_loss_weight=ft_loss_weight,
            aux_loader=aux_loader,
            alpha_schedule=alpha_schedule,
            alpha_warmup_epochs=alpha_warmup
        )


        # Save only if this trial is best so far (first trial will always save)
        try:
            current_best = trial.study.best_value   # fails on the first trial
        except ValueError:
            current_best = float("inf")

        torch.save(ft_student, f"trial_{trial.number}_ft_model_HTCas9_HT11_RfxCas13d_change_seq_sampled_2_tiger_iMeta_combined_to_predict_enAsCas12a_HF1_10_trained_loss_scaling_staged_curriculum_mixer.pt")


        # Save only if this trial is best so far (avoids writing every time)
        if ft_val_mse <= current_best:
            torch.save(ft_student, f"best_ft_model_HTCas9_HT11_RfxCas13d_change_seq_sampled_2_tiger_iMeta_combined_to_predict_enAsCas12a_HF1_10_trained_loss_scaling_staged_curriculum_mixer.pt")
            trial.set_user_attr("saved_as", "best_ft_model_HTCas9_HT11_RfxCas13d_change_seq_sampled_2_tiger_iMeta_combined_to_predict_enAsCas12a_HF1_10_trained_loss_scaling_staged_curriculum_mixer.pt")
            
        trial.set_user_attr("ft_val_mse", float(ft_val_mse))

        ft_student.eval()

        X1_ht = load_branch1_data_enAsCas12a_HF1()   # 90x3
        X2_ht = load_branch2_data_enAsCas12a_HF1()   # 90x3
        X3_ht = load_branch3_data_enAsCas12a_HF1()   # 120
        (c_ei_ht, tbh_ei_ht, d_ei_ht, 
         c_ea_ht, tbh_ea_ht, d_ea_ht,
         y_true) = load_dna_reaction_data_enAsCas12a_HF1()
    
    
        graph_ht = CRNADuplexSubstructuresDataset(
            c_ei_ht, tbh_ei_ht, d_ei_ht, 
            c_ea_ht, tbh_ea_ht, d_ea_ht, 
            reaction_rates=y_true
        )

        np.random.seed(42)
        full_indices = np.arange(len(y_true))
        selected_indices = np.random.choice(len(full_indices), size=2913, replace=False)
        unseen_indices = np.setdiff1d(full_indices, selected_indices)
        
        X1_ht_unseen = X1_ht[unseen_indices]
        X2_ht_unseen = X2_ht[unseen_indices]
        X3_ht_unseen = X3_ht[unseen_indices]
        y_unseen        = np.array(y_true)[unseen_indices]
    
        graph_ht_unseen = Subset(graph_ht, unseen_indices)
    
        ds_unseen = HybridDataset(graph_ht_unseen, X1_ht_unseen, X2_ht_unseen, X3_ht_unseen, y_unseen)
        eval_unseen_loader = DataLoader(ds_unseen, batch_size=256, shuffle=True,
                                     collate_fn=hybrid_collate, num_workers=0, pin_memory=pin)

        spearman_list = []
        for i in range(10):
            if len(ds_unseen) < 500:
                print("Not enough unseen samples to draw 500 each time.")
                break
            idx_trial = np.random.choice(len(ds_unseen), size=500, replace=False)
            trial_subset = Subset(ds_unseen, idx_trial)
            trial_loader = DataLoader(trial_subset, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
            preds_trial = []
            labels_trial = []
            with torch.no_grad():
                for c,tbh,d,x1,x2,x3,y in trial_loader:
                    p = ft_student(c,tbh,d,x1,x2,x3).cpu().numpy()
                    preds_trial.append(p.item())
                    labels_trial.append(y.item())
            sp_corr, _ = spearmanr(labels_trial, preds_trial)
            spearman_list.append(sp_corr)
            print(sp_corr)
        avg_spearman = np.mean(spearman_list) if spearman_list else None       
        print(f"Average Spearman (student vs experimental ground truth) on unseen enAsCas12a_HF1 dataset: {avg_spearman}")

        return float(ft_val_mse)



    prepare_ft_cache()   # populate FT_CACHE once

    ft_study = optuna.create_study(direction="minimize")
    ft_study.optimize(ft_objective, n_trials=10, show_progress_bar=False)


    best_trial = ft_study.best_trial
    print("Best FT trial:", best_trial.number)
    print("Best val MSE:", best_trial.value)
    best_ft_params = ft_study.best_params
    print("Best FT params:", best_ft_params)

    best_ft_student = torch.load(f"best_ft_model_HTCas9_HT11_RfxCas13d_change_seq_sampled_2_tiger_iMeta_combined_to_predict_enAsCas12a_HF1_10_trained_loss_scaling_staged_curriculum_mixer.pt", weights_only=False)
    best_ft_student.eval()


    # ---- Evaluate best student on UNSEEN experimental dataset ----
    print("\n=== Evaluating best KD student on experimental (unseen) dataset ===")
    X1_ht = load_branch1_data_enAsCas12a_HF1()   # 90x3
    X2_ht = load_branch2_data_enAsCas12a_HF1()   # 90x3
    X3_ht = load_branch3_data_enAsCas12a_HF1()   # 120
    (c_ei_ht, tbh_ei_ht, d_ei_ht, 
     c_ea_ht, tbh_ea_ht, d_ea_ht,
     y_true) = load_dna_reaction_data_enAsCas12a_HF1()


    graph_ht = CRNADuplexSubstructuresDataset(
        c_ei_ht, tbh_ei_ht, d_ei_ht, 
        c_ea_ht, tbh_ea_ht, d_ea_ht, 
        reaction_rates=y_true
    )

    np.random.seed(42)
    full_indices = np.arange(len(y_true))
    selected_indices = np.random.choice(len(full_indices), size=2913, replace=False)
    unseen_indices = np.setdiff1d(full_indices, selected_indices)
    
    X1_ht_unseen = X1_ht[unseen_indices]
    X2_ht_unseen = X2_ht[unseen_indices]
    X3_ht_unseen = X3_ht[unseen_indices]
    y_unseen        = np.array(y_true)[unseen_indices]

    graph_ht_unseen = Subset(graph_ht, unseen_indices)

    ds_unseen = HybridDataset(graph_ht_unseen, X1_ht_unseen, X2_ht_unseen, X3_ht_unseen, y_unseen)
    eval_unseen_loader = DataLoader(ds_unseen, batch_size=256, shuffle=True,
                                 collate_fn=hybrid_collate, num_workers=0, pin_memory=pin)

    

    spearman_list = []
    for i in range(100):
        if len(ds_unseen) < 500:
            print("Not enough unseen samples to draw 500 each time.")
            break
        idx_trial = np.random.choice(len(ds_unseen), size=500, replace=False)
        trial_subset = Subset(ds_unseen, idx_trial)
        trial_loader = DataLoader(trial_subset, batch_size=1, shuffle=False, collate_fn=hybrid_collate)

        preds_trial = []
        labels_trial = []
        with torch.no_grad():
            for c,tbh,d,x1,x2,x3,y in trial_loader:
                p = best_ft_student(c,tbh,d,x1,x2,x3).cpu().numpy()
                preds_trial.append(p.item())
                labels_trial.append(y.item())
        sp_corr, _ = spearmanr(labels_trial, preds_trial)
        spearman_list.append(sp_corr)
        print(sp_corr)
    avg_spearman = np.mean(spearman_list) if spearman_list else None       
    print(f"Average Spearman (student vs experimental ground truth) on unseen enAsCas12a_HF1 dataset: {avg_spearman}")
